In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import matplotlib.pyplot as plt
import numpy as np

class DelayedRecallDataset:
    def __init__(self, vocab_size=50, seq_len=30, delay=15):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.delay = delay

    def sample(self, batch_size):
        x = torch.zeros(batch_size, self.seq_len, dtype=torch.long)
        y = torch.zeros(batch_size, dtype=torch.long)

        for b in range(batch_size):
            key = random.randint(1, self.vocab_size // 2)
            value = random.randint(self.vocab_size // 2 + 1, self.vocab_size - 1)

            noise = [random.randint(1, self.vocab_size - 1) for _ in range(self.delay)]
            query = key

            seq = [key, value] + noise + [query]
            seq = seq[: self.seq_len]
            x[b, : len(seq)] = torch.tensor(seq)
            y[b] = value

        return x, y


class SinusoidalPE(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


# -----------------------------
# SCR Recurrent Memory Transformer
# -----------------------------

class SCRTransformer(nn.Module):
    def __init__(self, vocab, d_model=64, heads=4, mem_size=8):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.pe = SinusoidalPE(d_model)
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.retention = nn.Linear(d_model, 1)
        self.mem_size = mem_size
        self.out = nn.Linear(d_model, vocab)

    def forward(self, x, memory=None):
        h = self.pe(self.emb(x))
        if memory is not None:
            h = torch.cat([memory, h], dim=1)

        attn_out, _ = self.attn(h, h, h)
        h = self.ln1(h + attn_out)
        h = self.ln2(h + self.ff(h))

        scores = torch.sigmoid(self.retention(h))  # (B,T,1)
        h = h * scores

        # memory selection
        importance = scores.squeeze(-1)
        topk = torch.topk(importance, k=min(self.mem_size, importance.size(1)), dim=1)
        memory = torch.gather(
            h, 1, topk.indices.unsqueeze(-1).expand(-1, -1, h.size(-1))
        )

        logits = self.out(h[:, -1])
        return logits, memory, scores


# -----------------------------
# Sliding Window Baseline
# -----------------------------

class SlidingWindowTransformer(nn.Module):
    def __init__(self, vocab, d_model=64, heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.pe = SinusoidalPE(d_model)
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, vocab)

    def forward(self, x):
        h = self.pe(self.emb(x))
        attn_out, _ = self.attn(h, h, h)
        h = self.ln1(h + attn_out)
        h = self.ln2(h + self.ff(h))
        return self.out(h[:, -1])


# -----------------------------
# Training Loop
# -----------------------------

def train(model, dataset, steps=500, lr=3e-4):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    accs = []

    for step in range(steps):
        x, y = dataset.sample(batch_size=32)
        opt.zero_grad()

        if isinstance(model, SCRTransformer):
            logits, _, scores = model(x)
        else:
            logits = model(x)

        loss = F.cross_entropy(logits, y)
        loss.backward()
        opt.step()

        acc = (logits.argmax(-1) == y).float().mean().item()
        accs.append(acc)

    return accs


# -----------------------------
# Run Experiment
# -----------------------------

dataset = DelayedRecallDataset(delay=20)
scr = SCRTransformer(vocab=50)
sw = SlidingWindowTransformer(vocab=50)

scr_acc = train(scr, dataset)
sw_acc = train(sw, dataset)

# -----------------------------
# Plot Accuracy
# -----------------------------

plt.figure()
plt.plot(scr_acc, label="SCR-RMT")
plt.plot(sw_acc, label="Sliding Window")
plt.legend()
plt.title("Delayed Recall Accuracy")
plt.xlabel("Training Step")
plt.ylabel("Accuracy")
plt.savefig("accuracy_comparison.png")
plt.close()